# 👑 Aria Voss — The Broken Empress | Voice Aging MVP

**Character**: Aria Voss — starts as an idealistic liberator, becomes a paranoid tyrant, ends as a hollow exile.

**Core Personality**: `a powerful, intense female ruler — magnetic and commanding, voice that can silence a room, emotional depth beneath iron control`

## 📈 Age Stages
| Stage | Age | Title | Voice Modifier |
|---|---|---|---|
| Youth | 19 | The Liberator | bright passionate idealism, voice ringing with conviction, young queen's certainty, fire and hope |
| Prime | 34 | The Conqueror | commanding and resonant, iron authority, warmth replaced by calculation, still magnetic but dangerous |
| Middle | 49 | The Tyrant | paranoid and clipped, trust dissolved, the fire turned to cold fury, isolation audible in every word |
| Elder | 67 | The Exile | hollow and quiet, the fire entirely gone, stripped of everything, occasional flicker of the young woman she was |

In [ ]:
# 🛠️ 1. Setup & Imports
!pip install -q git+https://github.com/huggingface/transformers accelerate soundfile librosa
!pip install -q qwen-tts

import os
import gc
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUTPUT_DIR = "/content/aria_voss_aging"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Created output directory at: {OUTPUT_DIR}")

In [ ]:
# 🎭 2. Character Data
CHARACTER = {
    "name": "Aria Voss",
    "personality_core": "a powerful, intense female ruler — magnetic and commanding, voice that can silence a room, emotional depth beneath iron control",
    "stages": [
        {
            "stage_name": "Youth (19) — The Liberator",
            "voice_modifier": "bright passionate idealism, voice ringing with conviction, young queen's certainty, fire and hope",
            "prefix": "aria_voss_youth",
            "lines": [
                "I did not take this throne for power. I took it because someone had to. And no one else was brave enough to try.",
                "They told me a woman couldn't rule. I've heard that argument. It's remarkably easy to disprove.",
                "The people have suffered long enough. That ends today. I swear it on everything I am."
            ]
        },
        {
            "stage_name": "Prime (34) — The Conqueror",
            "voice_modifier": "commanding and resonant, iron authority, warmth replaced by calculation, still magnetic but dangerous",
            "prefix": "aria_voss_prime",
            "lines": [
                "Loyalty is not given. It is earned and then maintained through fear and respect in equal measure.",
                "I have made decisions that cost lives. I made them anyway. That is what ruling actually means.",
                "Do not mistake my stillness for weakness. Everything I do is intentional. Everything."
            ]
        },
        {
            "stage_name": "Middle (49) — The Tyrant",
            "voice_modifier": "paranoid and clipped, trust dissolved, the fire turned to cold fury, isolation audible in every word",
            "prefix": "aria_voss_middle",
            "lines": [
                "Everyone who once called themselves my friend has become an enemy or a liability. I've stopped seeing a difference.",
                "I know what they say about me in the streets. I know everything. That is how I've survived this long.",
                "Power doesn't corrupt. It reveals. This is who I always was. The throne just removed the pretense."
            ]
        },
        {
            "stage_name": "Elder (67) — The Exile",
            "voice_modifier": "hollow and quiet, the fire entirely gone, stripped of everything, occasional flicker of the young woman she was",
            "prefix": "aria_voss_elder",
            "lines": [
                "I had a vision of what this world could be. I still believe in that vision. I just became the wrong person to build it.",
                "They call me the Broken Empress. I've had worse titles. And more accurate ones.",
                "If I had one thing to do differently — just one — I would have learned to trust someone. Anyone. Before it was too late."
            ]
        }
    ]
}

In [ ]:
# 🧠 3. Load Qwen3-TTS VoiceDesign Model
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading {model_id}...")

model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print("Model loaded successfully!")

In [ ]:
# 🎙️ 4. Generate Aging Stages
print(f"Generating all age stages for {CHARACTER['name']}...\n")

for stage in CHARACTER["stages"]:
    print(f"=== {stage['stage_name']} ===")
    instruct = f"{CHARACTER['personality_core']}, {stage['voice_modifier']}"
    print(f"Voice Design Prompt:\n{instruct}\n")
    
    for i, line in enumerate(stage["lines"], 1):
        filename = f"{stage['prefix']}_{i:02d}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        print(f"Line {i}: {line}")
        
        # Generate voice
        wavs, sr = model.generate_voice_design(line, "English", instruct)
        
        # Save and display
        sf.write(filepath, wavs[0], sr)
        display(Audio(filepath))
        
        gc.collect()
        torch.cuda.empty_cache()
    print("-" * 40)

In [ ]:
# 🎬 5. Life Story Montage
print("=== Life Story Montage ===")
montage_audio = []
script = []
current_sr = 24000

for stage in CHARACTER["stages"]:
    line = stage["lines"][0] # Pick Line 1 from each stage
    instruct = f"{CHARACTER['personality_core']}, {stage['voice_modifier']}"
    
    wavs, sr = model.generate_voice_design(line, "English", instruct)
    current_sr = sr
    
    # 2.5 seconds of silence gap
    silence = np.zeros(int(2.5 * sr), dtype=np.float32)
    
    montage_audio.extend([wavs[0], silence])
    script.append(f"[{stage['stage_name']}] {line}")

final_audio = np.concatenate(montage_audio)
montage_path = os.path.join(OUTPUT_DIR, "aria_voss_life_story.wav")
sf.write(montage_path, final_audio, current_sr)

print("\n📜 Reading Script:")
for s in script:
    print(s)

print("\n▶️ Play Montage:")
display(Audio(montage_path))

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 🎭 6. Emotional Range
target_line = "I know what they say about me in the streets. I know everything. That is how I've survived this long."
prime_modifier = CHARACTER["stages"][1]["voice_modifier"]
base_instruct = f"{CHARACTER['personality_core']}, {prime_modifier}"

emotions = [
    {
        "name": "Cold Fury",
        "mod": "[cold fury]"
    },
    {
        "name": "Barely Controlled Grief",
        "mod": "[barely controlled grief]"
    },
    {
        "name": "Eerie Calm",
        "mod": "[eerie calm]"
    }
]

print("=== Emotional Range ===")
print(f"Line: \"{target_line}\"\n")

for emo in emotions:
    print(f"▶️ Register: {emo['name']}")
    instruct = f"{base_instruct}, {emo['mod']}"
    
    wavs, sr = model.generate_voice_design(target_line, "English", instruct)
    
    filename = f"aria_voss_prime_{emo['name'].replace(' ', '_').lower()}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    sf.write(filepath, wavs[0], sr)
    display(Audio(filepath))
    
    gc.collect()
    torch.cuda.empty_cache()
    print()

In [ ]:
# 📦 7. Download Outputs
import shutil
from google.colab import files

print("Zipping outputs...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print("Downloading...")
files.download(f"{OUTPUT_DIR}.zip")